In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib
import os

# Load Dataset

In [ ]:
df = pd.read_csv("../data/dataset_pupuk.csv")
# df = pd.read_csv("../data/sintetis/dataset_pupuk.csv")
print(f"Dataset: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Dataset: 5000 baris, 12 kolom


,jam,soil_moisture,soil_temperature,air_temperature,air_humidity,nitrogen,fosfor,kalium,ec,plant_age,fase,recommendation
0,0,55.0,27.6,25.2,78.9,62.7,86.9,104.9,2.50,0,establishment,0
1,1,52.2,26.1,24.6,80.2,64.5,91.1,102.7,2.33,0,establishment,0
2,2,50.5,25.8,24.3,76.8,62.9,92.3,107.9,2.25,0,establishment,0
3,3,49.6,25.0,24.6,79.5,62.1,92.0,110.8,2.24,0,establishment,0
4,4,49.4,26.5,23.9,78.9,63.5,88.0,114.1,2.23,0,establishment,0


# Definisi fitur dan label

In [3]:
FEATURES = ["nitrogen", "fosfor", "kalium", "plant_age", "fase",
            "ec", "soil_moisture"]

LABEL_NAMES = {
    0: "Tidak perlu",
    1: "Urea/ZA",
    2: "SP-36",
    3: "KCl",
    4: "Urea/ZA + SP-36",
    5: "Urea/ZA + KCl",
    6: "SP-36 + KCl",
    7: "Urea/ZA + SP-36 + KCl",
    8: "NPK 15-15-15",
    9: "Kurangi pemupukan N",
    10: "Flush air (EC/nutrisi tinggi)",
}

X = df[FEATURES].values
y = df["recommendation"].values

# Cek Distribusi Label (KRITIS)

In [4]:
print(f"Jumlah kelas unik: {len(set(y))}")
print(f"\nDistribusi label:")
dist = pd.Series(y).value_counts().sort_index()
for val, count in dist.items():
    print(f"  {val} = {LABEL_NAMES.get(val, '?'):35s} {count:5d} ({count/len(y)*100:.1f}%)")

if len(set(y)) < 2:
    print("\n⚠️  BAHAYA: Cuma 1 kelas! Model gak bisa dilatih.")
    print("    Data kamu perlu variasi kondisi NPK/fase lebih banyak.")

Jumlah kelas unik: 10

Distribusi label:
  0 = Tidak perlu                          1401 (28.0%)
  1 = Urea/ZA                              1235 (24.7%)
  2 = SP-36                                 208 (4.2%)
  3 = KCl                                   506 (10.1%)
  4 = Urea/ZA + SP-36                        22 (0.4%)
  5 = Urea/ZA + KCl                        1519 (30.4%)
  6 = SP-36 + KCl                            32 (0.6%)
  7 = Urea/ZA + SP-36 + KCl                  22 (0.4%)
  8 = NPK 15-15-15                           25 (0.5%)
  9 = Kurangi pemupukan N                    30 (0.6%)


# Split Train/Test

In [5]:
# stratify butuh tiap kelas minimal 2 sampel; kalau error, hapus stratify
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
except ValueError:
    print("Stratify gagal (ada kelas <2 sampel), pakai split biasa.")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

print(f"Train: {len(X_train)} baris")
print(f"Test:  {len(X_test)} baris")

Train: 4000 baris
Test:  1000 baris


# Training Random Forest

In [6]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)
print("Training selesai.")

ValueError: could not convert string to float: 'pematangan'

# Cross Validation

In [ ]:
n_splits = min(5, pd.Series(y_train).value_counts().min())
if n_splits < 2:
    print("Data per kelas terlalu sedikit untuk cross-validation.")
else:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_weighted")
    print(f"CV F1 (weighted): {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

CV F1 (weighted): 1.0000 (+/- 0.0000)


# Evaluasi

In [ ]:
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")
print()
labels_present = sorted(set(y_test) | set(y_pred))
target_names = [LABEL_NAMES.get(l, str(l)) for l in labels_present]
print(classification_report(y_test, y_pred, labels=labels_present,
                            target_names=target_names, zero_division=0))

Accuracy: 1.0000
F1 (weighted): 1.0000

              precision    recall  f1-score   support

     Urea/ZA       1.00      1.00      1.00        47

    accuracy                           1.00        47
   macro avg       1.00      1.00      1.00        47
weighted avg       1.00      1.00      1.00        47



# Feature Importance

In [ ]:
importances = model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

print("Feature Importance:")
for idx in sorted_idx:
    bar = "█" * int(importances[idx] * 50)
    print(f"  {FEATURES[idx]:15s} {importances[idx]:.4f}  {bar}")

Feature Importance:
  soil_moisture   0.0000  
  ec              0.0000  
  fase            0.0000  
  plant_age       0.0000  
  kalium          0.0000  
  fosfor          0.0000  
  nitrogen        0.0000  


# Tes Prediksi

In [ ]:
sample = {
    "nitrogen": 40,
    "fosfor": 40,
    "kalium": 100,
    "plant_age": 61,
    "fase": 1,
    "ec": 2.0,
    "soil_moisture": 65,
}
X_sample = np.array([[sample[f] for f in FEATURES]])
pred = int(model.predict(X_sample)[0])
conf = float(model.predict_proba(X_sample)[0].max())

print(f"Prediksi: {pred} = {LABEL_NAMES.get(pred)}")
print(f"Confidence: {conf:.2%}")


Prediksi: 1 = Urea/ZA
Confidence: 100.00%


# Simpan Model

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/rf_pupuk.joblib")
print("Model tersimpan: models/rf_pupuk.joblib")

Model tersimpan: models/rf_pupuk.joblib
